# Chapter 40: Dimensionality Reduction and Representation

Synthetic NRG product data illustrate PCA, reconstruction, and responsible two-dimensional views.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.dimensionality import fit_scaler,apply_scaler,fit_pca,variance_summary,reconstruction_error,top_loadings
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(40);n=240
volume=rng.normal(0,1,n);premium=rng.normal(0,1,n)
x=np.column_stack([100+25*volume+rng.normal(0,5,n),30+9*volume+rng.normal(0,3,n),.18-.04*volume+rng.normal(0,.02,n),45+10*premium+rng.normal(0,4,n),.22+.05*premium+rng.normal(0,.02,n),12-2*volume+3*premium+rng.normal(0,1,n)])
names=['weekly_units','orders','stockout_rate','margin','promo_share','lead_days']
cut=180;centre,scale=fit_scaler(x[:cut]);z_train=apply_scaler(x[:cut],centre,scale);z_test=apply_scaler(x[cut:],centre,scale)
print(f'Train: {len(z_train)}; test: {len(z_test)}; features: {x.shape[1]}')


Train: 180; test: 60; features: 6


In [ ]:
model,scores=fit_pca(z_train);summary=variance_summary(model)
for i,(r,c) in enumerate(zip(summary['ratio'],summary['cumulative']),1): print(f'PC{i}: ratio={r:.3f} cumulative={c:.3f}')


PC1: ratio=0.551 cumulative=0.551
PC2: ratio=0.363 cumulative=0.914
PC3: ratio=0.032 cumulative=0.946
PC4: ratio=0.028 cumulative=0.974
PC5: ratio=0.016 cumulative=0.990
PC6: ratio=0.010 cumulative=1.000


In [ ]:
for i,items in enumerate(top_loadings(model,names,3)[:2],1): print('PC'+str(i),', '.join(f'{name}={value:.2f}' for name,value in items))
for k in [1,2,3,4,6]:
 m,_=fit_pca(z_train,k);recon=m.inverse_transform(m.transform(z_test));print(f'k={k} test_RMSE={reconstruction_error(z_test,recon):.3f}')


PC1 weekly_units=0.48, orders=0.47, lead_days=-0.47
PC2 margin=0.58, promo_share=0.56, stockout_rate=-0.30
k=1 test_RMSE=0.753
k=2 test_RMSE=0.287
k=3 test_RMSE=0.233
k=4 test_RMSE=0.167
k=6 test_RMSE=0.000


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(range(1,7),summary['ratio']);axes[0].plot(range(1,7),summary['cumulative'],'o-');axes[0].set(xlabel='Component',ylabel='Variance share',title='Scree and cumulative variance')
axes[1].scatter(scores[:,0],scores[:,1],c=volume[:cut],cmap='viridis',s=20);axes[1].set(xlabel='PC1',ylabel='PC2',title='Training scores, colour used only for audit')
fig.tight_layout();plt.show()


## Interpretation

Two components preserve most standardized variation in this synthetic case, but that does not prove that they preserve every signal needed for a later decision. The projection was fitted on training data only.


In [ ]:
# Practice: compare downstream validation results for two, three, and all components.
